In [1]:
from langchain_community.document_loaders import PyPDFLoader
import chromadb
from pathlib import Path

# 1. Setup paths and parameters
PDF_PATH = "./documents"
DB_DIR = "./chromadb"

# Initialize your Chroma vector store
chroma_client = chromadb.PersistentClient(path=DB_DIR)

collection = chroma_client.get_or_create_collection(name="religious_collection")
# Specify the directory path (use "." for current directory)
directory = Path(PDF_PATH)

# List all .pdf files in the directory
pdf_files = list(directory.glob("*.pdf"))

for afile in pdf_files:
    print(f"Processing file: {afile.name}")
    # 1. Load the PDF - PyPDFLoader automatically separates text per page
    loader = PyPDFLoader(str(afile))
    pages = loader.load_and_split()

    for apage in pages:
        page_text = str(apage.page_content)
        page_id = str(apage.metadata['source'])
        page_id = page_id + "_" + str(apage.metadata['page'])
        
        # 2. Add documents to the Chroma vector store
        collection.add(documents=[page_text], ids=[page_id])
        
    print(f"Loaded {len(pages)} pages from {afile.name}.")


C:\Users\VenkyJagannath\AppData\Local\Temp\ipykernel_13088\182995905.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Processing file: atma_bodha.pdf


C:\Users\VenkyJagannath\.cache\chroma\onnx_models\all-MiniLM-L6-v2\onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:07<00:00, 11.4MiB/s]


Loaded 106 pages from atma_bodha.pdf.
Processing file: bgita.pdf


Ignoring wrong pointing object 70 0 (offset 0)
Ignoring wrong pointing object 224 0 (offset 0)


Loaded 154 pages from bgita.pdf.
Processing file: bgita_1.pdf
Loaded 951 pages from bgita_1.pdf.
Processing file: tattva_bodha.pdf
Loaded 96 pages from tattva_bodha.pdf.


In [2]:
import chromadb
chroma_client = chromadb.PersistentClient(path=DB_DIR)
collections = chroma_client.list_collections()
print("Collections in ChromaDB:", collections)


collection = chroma_client.get_collection(name="religious_collection")

query  = "what is causal body?"
# 4. Perform a similarity search
results = collection.query(
    query_texts=[query], # Your search query
    n_results=10                           # Number of similar results to return
)

print(results["ids"])  # Print the IDs of the similar documents



Collections in ChromaDB: [Collection(name=religious_collection)]
[['documents\\tattva_bodha.pdf_40', 'documents\\tattva_bodha.pdf_38', 'documents\\bgita_1.pdf_699', 'documents\\atma_bodha.pdf_48', 'documents\\bgita.pdf_120', 'documents\\atma_bodha.pdf_29', 'documents\\tattva_bodha.pdf_52', 'documents\\bgita_1.pdf_749', 'documents\\bgita_1.pdf_441', 'documents\\bgita_1.pdf_918']]


In [3]:
from langchain_ollama import ChatOllama
from langchain_classic.chains import ConversationalRetrievalChain
from langchain_chroma import Chroma
vector_store = Chroma(collection_name="religious_collection", client=chroma_client)

def query_after_getting_matched_documents(user_query, ollama_model_name="granite4.1:3b"):
    # Create a retriever from the vector store getting top 10 similar documents
    retriever = vector_store.as_retriever(collection_name="religious_collection", search_type="similarity", search_kwargs={"k": 10})

    llm = ChatOllama(model=ollama_model_name, base_url=None)
    # ConversationalRetrievalChain wraps the LLM + retriever
    chain = ConversationalRetrievalChain.from_llm(llm=llm, retriever=retriever, return_source_documents=True)

    result = chain.invoke({"question": user_query, "chat_history":[]})
    print(result["answer"])
    matching_docs = result["source_documents"]
    print("Matching document IDs:")
    for doc in matching_docs:
        print(doc.id)

In [4]:
query = "What is causal body?" 
query_after_getting_matched_documents(query)

The causal body (Ātman‑Avidyā or “Causal Body”) refers to a subtle, non‑material aspect associated with an individual’s personality that persists between physical lifetimes in various forms of theistic beliefs—particularly within Hindu philosophy and esoteric traditions. Here are some key points about it:

1. **Definition and Origin**  
   - In Vedic literature (especially the Upanishads), the causal body is described as a subtle, immaterial substance composed primarily of ignorance (Avidyā) that remains after the dissolution of the physical body at death.  
   - It carries with it the impressions, desires, and tendencies accumulated during one’s life, which determine what kind of future body or rebirth an individual will inhabit.

2. **Relation to Transmigration**  
   - The causal body is intimately linked to the doctrine of transmigration (reincarnation). According to this view, after death, a person's causal body—along with its accumulated karmic impressions—transmigrates into a ne

In [5]:
query = "What is sthoola sharira?" 
query_after_getting_matched_documents(query)

Sthūla Sharīra (स्थूल शरीर) refers to the gross or physical body in Hindu philosophy, particularly in the context of Samkhya and Yoga traditions. It encompasses all that which can be perceived through the five senses—touch, taste, sight, hearing, and smell—and is composed of matter (sthula stanya). This material body includes the organs of sense, organs of action, mind, intellect, ego, and prakriti (nature or elements) that govern its functions. In contrast, the subtle body (pṛthāgaśīriṇa sharīra) consists of the finer aspects like the mind, intellect, ego, and their impressions, which are not perceptible through the senses but influence the functioning of the sthūla sharīra.
Matching document IDs:
documents\bgita.pdf_2
documents\bgita_1.pdf_563
documents\bgita_1.pdf_9
documents\bgita_1.pdf_557
documents\bgita_1.pdf_518
documents\bgita_1.pdf_460
documents\bgita.pdf_129
documents\bgita_1.pdf_905
documents\bgita.pdf_25
documents\bgita_1.pdf_730


In [6]:
query = "What is krishna's advice to us to get free from karma?" 
query_after_getting_matched_documents(query)

Krishna advises that the way to be free from karma (the cycle of cause and effect) is to perform all actions—whether they are bound by desire or not—as an offering to Him. In other words, one should engage in activities with a pure, detached spirit of devotion, treating every action as if it were a sacrifice offered to Krishna rather than being driven solely by personal desires or the expectation of worldly rewards. By doing so, actions cease to bind us because they are performed for the purpose of serving Him and not merely for self-interest or material gain. This detachment transforms ordinary duties into spiritual service, allowing one to transcend the illusory effects (karma) that bind the soul in a cycle of birth and rebirth.
Matching document IDs:
documents\bgita.pdf_142
documents\bgita.pdf_94
documents\bgita.pdf_10
documents\bgita_1.pdf_630
documents\bgita.pdf_6
documents\bgita.pdf_149
documents\bgita_1.pdf_150
documents\bgita_1.pdf_208
documents\bgita.pdf_47
documents\bgita.pdf

In [7]:
query = "What is lotus leaf analogy given to us?" 
query_after_getting_matched_documents(query)

The lotus leaf analogy, as presented in the Bhagavad Gita (Chapter 2, verse 13) and other spiritual texts, is used to illustrate several important concepts related to purity, resilience, and divine beauty. The key points of this analogy are:

1. **Purity Amidst Dirt**: Just as a lotus flower remains clean and pure despite growing in muddy water, the soul (or one's true self) can remain unaffected by external impurities or circumstances that surround it.

2. **Resilience to Adversity**: The lotus leaf is smooth and unblemished even when covered with mud particles. Similarly, a person who remains steadfast in their spiritual practice can maintain inner purity and clarity despite facing worldly challenges or moral corruption around them.

3. **Divine Beauty and Attraction**: The lotus flower's beauty and fragrance are not compromised by its growth in dirty water; rather, it attracts attention due to its pristine appearance. This symbolizes how the divine qualities within an individual (or

In [8]:
query = "What is the difference between the northern path and southern path of the soul?" 
query_after_getting_matched_documents(query)

The distinction you’re referring to—often called the **Northern Path** (also known as the **Brahma Varga**) versus the **Southern Path** (or **Jiva Varga**)—is a concept found in certain philosophical traditions, particularly within Indian spiritual thought and Vedānta. While these terms aren’t universally standardized across all texts, they generally describe different aspects of the relationship between the individual soul (**jīva**) and the ultimate reality (**Brahman** or **Ātman**). Here’s a high-level overview:

### Northern Path (Brahma Varga)

1. **Ultimate Identity**: The Northern Path posits that the individual soul (*jīva*) is ultimately identical with Brahman—the absolute, unchanging reality. This view emphasizes **monism**, asserting that everything arises from and returns to this one universal principle.
   
2. **Emphasis on Transcendence**: It stresses the transcendental nature of the Ātman; it transcends all modifications (including both pleasure and pain) because it is

In [9]:
query = "What happens to arjuna after seeing the vishwaroopa of krishna?" 
query_after_getting_matched_documents(query)

After witnessing Krishna's Vishvaroopa (universal form) for the first time, Arjuna’s reaction is one of profound awe and reverence. His hair stands on end, his body trembles with ecstasy, and he begins to offer obeisances to Krishna with folded hands, indicating a deep sense of humility and devotion. This experience transforms their relationship from that of casual friends or companions in arms to one marked by intense spiritual connection and respect for the Supreme Personality of Godhead.

Krishna’s revelation not only dispels Arjuna's initial feelings of helplessness and moral conflict but also provides him with a divine perspective on the significance and inevitability of the battle. It reassures him that engaging in this war is part of dharma (righteous duty) and that victory over his enemies will ultimately benefit society, preserve righteousness, and fulfill their rightful claims to rule.

In essence, seeing Krishna's Vishvaroopa instills in Arjuna a renewed sense of purpose, co

In [10]:
query = "Why is arjuna lamenting in the beginning?" 
query_after_getting_matched_documents(query)

Arjuna is lamenting at the beginning primarily because he faces an extremely difficult moral dilemma: he is about to engage in a battle against his own relatives, the Kauravas led by Duryodhana. This situation creates deep emotional anguish and conflict for him:

1. **Duty vs. Emotion**: Arjuna feels immense pressure as a kñatriya (warrior) to fulfill his duty of protecting dharma (righteousness) and fighting in the war, even though it means he will likely be killing his own kin.

2. **Fear of Injustice**: He is troubled by the idea that victory for one side could lead to suffering and loss on a massive scale, particularly for his brothers who might fall as casualties on the battlefield.

3. **Sense of Responsibility**: Arjuna carries heavy responsibilities not just to himself but also to his kingdom and people. The outcome of this battle would have significant consequences for them, which adds to his distress.

4. **Inner Conflict**: There’s an internal conflict between what he believ

In [11]:
query = "How does the atma reveal itself in deep meditation? What analogy does Shankaracharya give us for this?" 
query_after_getting_matched_documents(query)

In deep meditation, according to Adi Shankara, the Atman (the Supreme Self) reveals itself by transcending its false identification with individual ego and external appearances. This realization comes when one moves beyond ordinary perceptions—such as seeing oneself primarily through thoughts of “I am a separate entity” or material objects—to directly experience the underlying reality of pure consciousness, devoid of any form or limitation.

**Analogy Given by Shankaracharya:**

Shankara compares this revelation to recognizing that what was originally perceived as a “snake” (the ego’s sense of self) is actually just a “rope.” In his verse 27:

- **U‹ÑxÉmÉïuÉSÉiqÉÉlÉÇ eÉÏuÉÇ ¥ÉÉiuÉÉ pÉrÉÇ** – This represents the mistaken identification of the individual self (jeevan) with an egoic sense, leading to fear.
  
- **lÉÉWÇû eÉÏuÉÈ mÉUÉiqÉåÌiÉ ¥ÉÉiÉÇ cÉåÍ³ÉpÉïrÉÉå pÉuÉåiÉç** – Here he describes how, just as one can see that a rope is not a snake without fear, the true Self (paraatmaa) remains 

In [12]:
query = "How are the prana vayus created?" 
query_after_getting_matched_documents(query)

According to Vedic cosmology, specifically as described in ancient texts such as the *Yoga Sutras of Patanjali* and related philosophical literature (e.g., the *Hatha Yoga Pradipaka*), the five prana vayus—*Prāṇa*, *Vyāna*, *Udana*, *Pasūta*, and *Aṣṭha*—are considered to be subtle manifestations of the universal energy known as **Sattva** (the quality associated with knowledge, harmony, and purity). Their creation and development are understood through a hierarchical evolution that progresses from subtlety toward greater grossness:

1. **Prāṇa (Life Force)** – This is the most fundamental prana vayu and is directly linked to the subtlest form of Sattva. It represents pure consciousness, vitality, and the ability to sustain life at a cellular level.

2. **Vyāna (Spread/Expansion)** – Vyāna arises from Prāṇa by incorporating additional qualities—namely, *Rajas* (the quality associated with action and transformation). Vyāna expands the reach of energy throughout the body, enabling bodily

In [16]:
query = "What are the parts making up the subtle body?" 
query_after_getting_matched_documents(query)

The subtle body (Sukshma Sharira) comprises 17 distinct components, as defined in Bhagavad Gita Chapter 3, Verse 2. These components include:

1. **Apanchi krita pancha maha bhootaih kritam** – composed of the five gross elements that have not undergone grossification.
2. **Satkarma janyam** – born from past good actions (Karmas) accumulated over many lifetimes.
3. **Sukha duhkha aadi** – responsible for experiencing experiences such as joy and sorrow.
4. **Bhoga saadhanam** – the instrument through which these experiences are experienced.
5. **Pancha jnaana indriyaani** – five sense organs (eyes, ears, nose, tongue, and skin).
6. **Pancha karma indriyaani** – five organs of action (hands, feet, private parts, speech, and the mind).
7. **Pancha praana aadayah** – five Pranas (vital airs) such as prana, apana, vyana, samana, and udana.
8. **Manah cha ekam** – one mind that governs all these elements.
9. **Buddhih cha ekaa** – the single intellect that interprets experiences and guides a

In [17]:
query = "What is gross body?" 
query_after_getting_matched_documents(query)

The gross body refers to the physical, tangible, and visible aspects of an individual that can be perceived through the senses—such as sight, touch, hearing, taste, and smell. It encompasses all the material components of a person: bones, muscles, organs, tissues, blood, skin, hair, nails, etc., which constitute the overall form and structure of the living being in the material (jala, apas, vyapar) world.

In contrast to the subtle body (which includes the mind, intellect, ego, and other non-physical elements), the gross body is subject to decay, aging, illness, and ultimately death. It is constantly changing and undergoing processes such as growth, maturation, senescence, and eventual dissolution back into matter. The gross body interacts with the external environment through senses and can be influenced by various factors like nutrition, exercise, disease, etc., but it does not possess inherent spiritual qualities or consciousness beyond its physical functions.
Matching document IDs:

In [18]:
query = "Can you please summarize gita chapter 2?" 
query_after_getting_matched_documents(query)

Chapter 2 of the Bhagavad Gītā is titled “Ānand‑Muktaṁ Sva‑Padañjām” (The Verse Which Leads to Eternal Bliss). In this chapter, Lord Krishna addresses Arjuna’s doubts and introduces several foundational concepts:

1. **Immortality of the Soul (Atman):**  
   - Krishna emphasizes that every living being possesses an eternal soul (atma) which is immortal and distinct from the body. This immortality is independent of birth or death.

2. **Identity with Brahman:**  
   - He explains that the individual self (jiva) is ultimately one with the Supreme Soul, Brahman—i.e., “Tat Twam Asi” (“That Thou Art”). Understanding this identity dissolves all distinctions between the jiva and Brahman.

3. **Nature of Ego‑Thoughts:**  
   - Krishna points out that egoistic thoughts (citta-vritti) are transient mental modifications arising from ignorance (avidyā). These modifications give rise to bondage, suffering, and illusion.

4. **Path to Liberation (Mokṣa):**  
   - To overcome the cycle of birth and d